# W5: Your first Replicate API call

**Goal:** connect a prompt to an API call, display the image, and save it.
Open this notebook in Jupyter and run cells in order with **Shift+Enter**.
This introduction has one model and no game controls.

We start in **offline practice mode**, using a labelled test diagram rather than an AI-generated image.
Switch to live mode with the instructor when ready. Each live request uses your own Replicate account and may incur a charge.

Initial model: [FLUX.1 schnell](https://replicate.com/black-forest-labs/flux-schnell).
The instructor can replace it after checking its inputs.

Run the installation cell below if the libraries are missing. It needs internet access but makes no generation request.
Once installed, skip it during offline practice. Restart the kernel if Jupyter asks you to.
See [README.md](README.md) for shared setup instructions.

In [ ]:
%pip install -q replicate Pillow

## 1. Choose practice or live mode

Keep `LIVE = False` for an offline walkthrough. No API token or account request is needed.


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
from uuid import uuid4
import json
from PIL import Image as PILImage, ImageDraw
from IPython.display import display, Image, FileLink

LIVE = False
MODEL = "black-forest-labs/flux-schnell"


## 2. Connect privately

In live mode, get your token from [Replicate](https://replicate.com/account/api-tokens)
and paste it only into the hidden input below. Never paste it into a code cell.
The token stays in kernel memory. Restart the kernel when finished.

The client uses a 120-second HTTP timeout. This is not a two-minute limit on the whole generation:
waiting for a prediction can involve several requests.

In [ ]:
client = None

if LIVE:
    import replicate
    from getpass import getpass

    token = getpass("Replicate API token (hidden): ").strip()
    if not token:
        raise ValueError("Enter your API token, or switch back to practice mode.")
    client = replicate.Client(api_token=token, timeout=120)
    del token
    
    print("Client ready. No generation has been requested.")
else:
    print("Offline practice: no API connection.")


## 3. Write the prompt

This dictionary corresponds to the model's web UI inputs. `prompt` describes the scene;
the other entries request one square PNG. Other models may use different inputs.


In [ ]:
prompt = "A cyclist in a yellow raincoat crosses a foggy stone bridge at dawn, wide shot, muted colours."
inputs = {
    "prompt": prompt,
    "num_outputs": 1,
    "aspect_ratio": "1:1",
    "output_format": "png",
}


## 4. Create a prediction and wait for it

The function looks up the model's latest version, creates a prediction, prints its ID, and waits.
It returns image URLs only after the prediction succeeds. A failed or cancelled prediction produces an explanatory message instead of a "ready" result.

In live mode, type **GENERATE** to send one paid request. Rerunning the generation cell can make another request.
If you interrupt the wait, check the printed prediction ID in Replicate before retrying.
Displaying or saving below never generates again.

In [ ]:
class GenerationError(Exception):
    """A prediction finished without a usable image."""


def generate_image(inputs, client, MODEL):
    model = client.models.get(MODEL)
    if model.latest_version is None:
        raise GenerationError("This model has no available version. Check the model with the instructor.")
    version_id = model.latest_version.id

    prediction = client.predictions.create(version=version_id, input=inputs)
    print("Prediction ID:", prediction.id)
    prediction.wait()

    if prediction.status != "succeeded":
        raise GenerationError(
            f"Prediction {prediction.id} ended with status '{prediction.status}'. "
            "Check its details in Replicate before trying again."
        )
    urls = prediction.output
    if not isinstance(urls, list) or not urls or not isinstance(urls[0], str) or not urls[0].startswith("https://"):
        raise GenerationError(
            f"Prediction {prediction.id} returned no usable image URL. "
            "Check the model's output format with the instructor."
        )
    return urls

In [ ]:
output = None
saved_image = None
run_folder = None
image_bytes = None
run_context = None
if not prompt.strip():
    raise ValueError("Write a prompt first.")
if LIVE:
    if client is None:
        raise ValueError("Run the connection cell first.")
    if input("One paid image request. Type GENERATE to continue: ") == "GENERATE":
        try:
            output = generate_image(inputs, client, MODEL)
            run_context = {"model": MODEL, "inputs": dict(inputs), "mode": "live"}
            print("Generation returned. Run the next cell to save it.")
        except GenerationError as error:
            print(str(error))
        except Exception:
            print("Request failed here. Check token, credit, model inputs and your Replicate predictions before retrying. A prediction may already exist. No automatic retry was made.")
else:
    canvas = PILImage.new("RGB", (640, 400), "#edf2f4")
    draw = ImageDraw.Draw(canvas)
    draw.text((24, 20), "OFFLINE PRACTICE / NOT AI GENERATED", fill="black")
    draw.rectangle((60, 140, 220, 300), fill="#e76f51")
    draw.ellipse((280, 140, 440, 300), fill="#2a9d8f")
    buffer = BytesIO()
    canvas.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    run_context = {"model": None, "inputs": dict(inputs), "mode": "practice"}
    print("Practice diagram ready. It does not respond to your prompt.")
if run_context is not None:
    run_context["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    run_folder = Path("outputs") / "basics" / uuid4().hex[:12]

## 5. Download the result and save it locally

With the prediction API used here, `output[0]` is the first image's URL.
`urlretrieve(url, target)` downloads that file. Offline practice saves the diagram's bytes directly.

Save promptly rather than relying on the hosted URL. If downloading fails, retry this cell,
not the paid generation cell. Each generation has its own output folder.
The saved prompt record uses the mode captured when the result was created.

In [ ]:
from urllib.request import urlretrieve

saved_image = None
if run_context is None:
    print("No result to save. Check the generation cell.")
else:
    try:
        run_folder.mkdir(parents=True, exist_ok=True)
        target = run_folder / "image.png"
        if run_context["mode"] == "live":
            if not output:
                raise ValueError("No image URL is available.")
            urlretrieve(output[0], target)
        elif image_bytes is not None:
            with open(target, "wb") as f:
                f.write(image_bytes)
        else:
            raise ValueError("No image bytes are available.")
        (run_folder / "prompt.json").write_text(json.dumps(run_context, indent=2), encoding="utf-8")
        saved_image = target
        print("Saved image and prompt record in", run_folder)
    except Exception:
        print("Could not download or save the image. Check the output URL and local folder permissions. Retry this cell before making another paid request.")

## 6. Display and download

These links download files from your Jupyter server to your computer. No new API call is made.


In [ ]:
if saved_image is not None:
    display(Image(filename=str(saved_image), width=480))
    display(FileLink(str(saved_image)))
    display(FileLink(str(run_folder / "prompt.json")))
else:
    print("Save an image first.")


## Try one change

In live mode, change one detail in the prompt, rerun steps 3 to 6, and compare both saved outputs.
What did you intend to change? What changed unexpectedly? A single pair is an observation,
not proof that a prompt is always better. In practice mode, rehearse saving only; the diagram stays the same.

Next: [Artistic Telephone](02_artistic_telephone.ipynb).
Keep prompt records private during the game. Never submit a token.

Official references: [Python guide](https://replicate.com/docs/get-started/python/),
[model inputs](https://replicate.com/black-forest-labs/flux-schnell/api),
[file outputs](https://replicate.com/docs/topics/predictions/output-files).
